# RAGAS Evaluation Framework [Step 2 - Automated Evaluation at Scale]

> **MLCourse - Agentic AI - RAG Evaluation**

RAGAS (Retrieval Augmented Generation Assessment) is the standard open-source
framework for evaluating RAG systems. It provides pre-built metrics, dataset
creation tools, and evaluation pipelines.

In this notebook we:
1. Build a RAG pipeline over Alice in Wonderland
2. Create a RAGAS-compatible evaluation dataset
3. Run the full RAGAS evaluation suite
4. Interpret the results and compare strategies

In [1]:
# ## 1. Setup and Environment

In [2]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

print("[setup] RAGAS evaluation framework notebook initialized")

[setup] RAGAS evaluation framework notebook initialized


In [3]:
# ## 2. Install and Import RAGAS
# RAGAS provides faithfulness, answer relevance, context precision, and
# context recall out of the box.

In [4]:
try:
    import ragas
    from ragas import evaluate, EvaluationDataset, SingleTurnSample
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
    )
    RAGAS_AVAILABLE = True
    print(f"[ragas] RAGAS version: {ragas.__version__}")
except ImportError as e:
    RAGAS_AVAILABLE = False
    print(f"[ragas] RAGAS not installed: {e}")
    print("[ragas] Run: pip install ragas")

[ragas] RAGAS not installed: No module named 'langchain_community.chat_models.vertexai'
[ragas] Run: pip install ragas


In [5]:
# ## 3. Initialize the LLM

In [6]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("[llm] ChatOllama ready:", llm.model)

[llm] ChatOllama ready: llama3.1:8b


In [7]:
# ## 4. Build the RAG Pipeline
# We build a FAISS-based RAG pipeline over Alice in Wonderland chapters.

In [8]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
import re

ALICE_PATH = Path(r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt")
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")[:30_000]

# Split by chapters for better retrieval
chapter_pattern = r"^CHAPTER [IVX]+\."
chapter_starts = list(re.finditer(chapter_pattern, raw_text, flags=re.MULTILINE))

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
documents = []

for i, match in enumerate(chapter_starts):
    end = chapter_starts[i + 1].start() if i + 1 < len(chapter_starts) else len(raw_text)
    chapter_text = raw_text[match.start():end]
    chapter_num = i + 1
    for j, chunk in enumerate(splitter.split_text(chapter_text)):
        documents.append(Document(
            page_content=chunk,
            metadata={"chapter": chapter_num, "chunk_id": f"c{chapter_num}_{j}"}
        ))

print(f"[rag] Built {len(documents)} chunks across {len(chapter_starts)} chapters")

[rag] Built 77 chunks across 3 chapters


In [9]:
# Create vector store and RAG chain.

In [10]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    parts = []
    for doc in docs:
        parts.append(f"[{doc.metadata.get('chunk_id', '?')}] {doc.page_content}")
    return "\n\n".join(parts)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about Alice in Wonderland using ONLY the provided context. "
     "Cite chunk IDs when referencing specific information."),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("[rag] RAG chain ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[rag] RAG chain ready


In [11]:
# ## 5. Create Ground Truth Reference Answers
# RAGAS needs reference answers to compute context recall and other metrics.
# We define them manually for our test questions.

In [12]:
test_questions = [
    {
        "question": "Why did Alice fall down the rabbit hole?",
        "reference_answer": "Alice fell down the rabbit hole because she was following a white rabbit and became curious about where it went. She leaned over and fell into a deep hole.",
        "reference_contexts": [
            "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do. She had peeped into the book her sister was reading, but it had no pictures or conversations in it.",
            "In another moment down went Alice after it, never once considering how in the world she was to get out again.",
        ],
    },
    {
        "question": "What did the Cheshire Cat tell Alice about madness?",
        "reference_answer": "The Cheshire Cat told Alice that we are all mad here, meaning that madness is normal in Wonderland. He said he is mad and she is mad.",
        "reference_contexts": [
            "We are all mad here. I am mad. You are mad.",
            "How do you know I am mad? said Alice. You must be, said the Cat, or you would not have come here.",
        ],
    },
    {
        "question": "What was the Queen of Hearts obsessed with?",
        "reference_answer": "The Queen of Hearts was obsessed with beheading people. She repeatedly shouted Off with his head! over minor offenses.",
        "reference_contexts": [
            "The Queen of Hearts, she made some tarts, all on a summer day.",
            "Off with his head! was her frequent command.",
        ],
    },
    {
        "question": "What happened at the croquet game?",
        "reference_answer": "The croquet game was chaotic because the mallets were live flamingos and the balls were hedgehogs. The Queen of Hearts kept ordering executions.",
        "reference_contexts": [
            "The soldiers used their红 mázzles for arches, through which the players had to march.",
            "The croquet ground was all ridged and furrowed with hills and valleys.",
        ],
    },
    {
        "question": "Who is Alice's sister?",
        "reference_answer": "Alice's sister was reading a book without pictures or conversations on the riverbank. She is older and more serious than Alice.",
        "reference_contexts": [
            "Alice was beginning to get very tired of sitting by her sister on the bank.",
            "Her sister was reading a book without pictures or conversations in it.",
        ],
    },
]

print(f"[eval] Defined {len(test_questions)} test questions with reference answers")

[eval] Defined 5 test questions with reference answers


In [13]:
# ## 6. Build RAGAS Evaluation Dataset
# RAGAS uses `SingleTurnSample` objects that contain the question, answer,
# retrieved contexts, and ground truth.

In [14]:
if RAGAS_AVAILABLE:
    dataset_samples = []

    for item in test_questions:
        # Get RAG answer and retrieved contexts
        answer = rag_chain.invoke(item["question"])
        docs = retriever.invoke(item["question"])
        contexts = [doc.page_content for doc in docs]

        sample = SingleTurnSample(
            user_input=item["question"],
            response=answer,
            retrieved_contexts=contexts,
            reference=item["reference_answer"],
            reference_contexts=item["reference_contexts"],
        )
        dataset_samples.append(sample)
        print(f"  Built sample for: {item['question'][:50]}...")

    eval_dataset = EvaluationDataset(samples=dataset_samples)
    print(f"\n[dataset] Created RAGAS evaluation dataset with {len(eval_dataset)} samples")
else:
    print("[ragas] Skipping dataset creation (RAGAS not available)")

[ragas] Skipping dataset creation (RAGAS not available)


In [15]:
# ## 7. Run RAGAS Evaluation
# We run the full RAGAS evaluation suite: faithfulness, answer relevancy,
# context precision, and context recall.

In [16]:
if RAGAS_AVAILABLE:
    metrics = [
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
    ]

    print("[eval] Running RAGAS evaluation...")
    print("[eval] Metrics:", [m.name for m in metrics])
    print()

    result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
    )

    print("[eval] Evaluation complete!")
    print()
    print(result)
else:
    print("[ragas] Skipping evaluation (RAGAS not available)")
    print("[ragas] Install with: pip install ragas")

[ragas] Skipping evaluation (RAGAS not available)
[ragas] Install with: pip install ragas


In [17]:
# ## 8. Interpret RAGAS Scores
# RAGAS scores range from 0 to 1. Here is how to interpret them.

In [18]:
print("RAGAS Score Interpretation")
print("=" * 55)
print()
print("Faithfulness (0-1)")
print("  Measures if the answer is grounded in the retrieved context.")
print("  1.0 = every claim is supported by context")
print("  < 0.5 = the model is hallucinating significantly")
print()
print("Answer Relevancy (0-1)")
print("  Measures if the answer addresses the question.")
print("  1.0 = answer is perfectly aligned with the question")
print("  < 0.5 = answer is off-topic or incomplete")
print()
print("Context Precision (0-1)")
print("  Measures if the retrieved contexts are relevant and ranked well.")
print("  1.0 = all retrieved chunks are relevant, most relevant first")
print("  < 0.5 = many irrelevant chunks retrieved")
print()
print("Context Recall (0-1)")
print("  Measures if the retrieved contexts cover the reference answer.")
print("  1.0 = all information in reference is present in contexts")
print("  < 0.5 = significant information gaps in retrieval")

RAGAS Score Interpretation

Faithfulness (0-1)
  Measures if the answer is grounded in the retrieved context.
  1.0 = every claim is supported by context
  < 0.5 = the model is hallucinating significantly

Answer Relevancy (0-1)
  Measures if the answer addresses the question.
  1.0 = answer is perfectly aligned with the question
  < 0.5 = answer is off-topic or incomplete

Context Precision (0-1)
  Measures if the retrieved contexts are relevant and ranked well.
  1.0 = all retrieved chunks are relevant, most relevant first
  < 0.5 = many irrelevant chunks retrieved

Context Recall (0-1)
  Measures if the retrieved contexts cover the reference answer.
  1.0 = all information in reference is present in contexts
  < 0.5 = significant information gaps in retrieval


In [19]:
# ## 9. Compare Two RAG Strategies
# We compare a basic RAG chain against one with query expansion.

In [20]:
def basic_retrieval(query):
    """Simple retriever with k=4."""
    return retriever.invoke(query)

def expanded_retrieval(query):
    """Retrieve with original + expanded query."""
    expansion_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Generate 2 alternative phrasings of the following question. "
         "Return only the questions, one per line."),
        ("user", "Question: {query}")
    ])
    expanded = (expansion_prompt | llm | StrOutputParser()).invoke({"query": query})

    all_docs = retriever.invoke(query)
    for alt_q in expanded.strip().split("\n"):
        alt_q = alt_q.strip()
        if alt_q:
            all_docs.extend(retriever.invoke(alt_q))

    # Deduplicate by chunk_id
    seen = set()
    unique_docs = []
    for doc in all_docs:
        cid = doc.metadata.get("chunk_id", id(doc))
        if cid not in seen:
            seen.add(cid)
            unique_docs.append(doc)

    return unique_docs[:6]

In [21]:
# Build both RAG chains.

In [22]:
rag_prompt2 = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about Alice in Wonderland using ONLY the provided context."),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

basic_chain = (
    {"context": RunnableLambda(basic_retrieval) | format_docs, "question": RunnablePassthrough()}
    | rag_prompt2 | llm | StrOutputParser()
)

expanded_chain = (
    {"context": RunnableLambda(expanded_retrieval) | format_docs, "question": RunnablePassthrough()}
    | rag_prompt2 | llm | StrOutputParser()
)

print("[compare] Built basic and expanded RAG chains")

[compare] Built basic and expanded RAG chains


In [23]:
# Run RAGAS on both strategies.

In [24]:
if RAGAS_AVAILABLE:
    strategies = {
        "basic": basic_chain,
        "expanded": expanded_chain,
    }

    comparison_results = {}
    for name, chain in strategies.items():
        samples = []
        for item in test_questions:
            answer = chain.invoke(item["question"])
            docs = retriever.invoke(item["question"])
            sample = SingleTurnSample(
                user_input=item["question"],
                response=answer,
                retrieved_contexts=[d.page_content for d in docs],
                reference=item["reference_answer"],
                reference_contexts=item["reference_contexts"],
            )
            samples.append(sample)

        ds = EvaluationDataset(samples=samples)
        res = evaluate(dataset=ds, metrics=metrics)
        comparison_results[name] = res
        print(f"\n[{name}] Strategy results:")
        print(res)

    # Summary comparison
    print("\n" + "=" * 60)
    print("Strategy Comparison Summary")
    print("=" * 60)
    for name, res in comparison_results.items():
        print(f"\n  {name}:")
        for metric_name in ["faithfulness", "answer_relevancy",
                            "context_precision", "context_recall"]:
            if metric_name in res:
                print(f"    {metric_name}: {res[metric_name]:.4f}")
else:
    print("[ragas] Skipping comparison (RAGAS not available)")

[ragas] Skipping comparison (RAGAS not available)


In [25]:
# ## 10. Manual Evaluation Fallback
# If RAGAS is not installed, we can still evaluate using our custom metrics
# from notebook 01.

In [26]:
def manual_evaluate(question, chain, retriever):
    """Evaluate using manual metrics when RAGAS is unavailable."""
    answer = chain.invoke(question)
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # Simple keyword-based faithfulness check
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())
    overlap = len(answer_words & context_words) / max(len(answer_words), 1)

    # Simple length-based relevance (longer answers tend to be more relevant)
    length_score = min(len(answer) / 200, 1.0)

    # Context count
    context_score = min(len(docs) / 4, 1.0)

    return {
        "question": question,
        "answer": answer[:150],
        "word_overlap": overlap,
        "length_score": length_score,
        "context_score": context_score,
    }

In [27]:
# Run manual evaluation if RAGAS is not available.

In [28]:
if not RAGAS_AVAILABLE:
    print("Running manual evaluation fallback:")
    print("=" * 60)
    for item in test_questions:
        res = manual_evaluate(item["question"], rag_chain, retriever)
        print(f"Q: {res['question'][:50]}")
        print(f"  Word overlap: {res['word_overlap']:.4f}")
        print(f"  Length score: {res['length_score']:.4f}")
        print(f"  Context score: {res['context_score']:.4f}")
        print()

Running manual evaluation fallback:


Q: Why did Alice fall down the rabbit hole?
  Word overlap: 0.6757
  Length score: 1.0000
  Context score: 1.0000



Q: What did the Cheshire Cat tell Alice about madness
  Word overlap: 0.5000
  Length score: 0.7450
  Context score: 1.0000



Q: What was the Queen of Hearts obsessed with?
  Word overlap: 0.2500
  Length score: 0.7050
  Context score: 1.0000



Q: What happened at the croquet game?
  Word overlap: 0.5849
  Length score: 1.0000
  Context score: 1.0000



Q: Who is Alice's sister?
  Word overlap: 0.5143
  Length score: 1.0000
  Context score: 1.0000



In [29]:
# ## 11. Exporting Evaluation Results

In [30]:
# Save evaluation results to JSON for later analysis.
output_path = Path(r"D:\projects\python\MLCourse\03_agentic_ai\29_rag_evaluation")
output_path.mkdir(exist_ok=True)

export_data = {
    "notebook": "02_ragas_framework",
    "test_questions": [item["question"] for item in test_questions],
    "ragas_available": RAGAS_AVAILABLE,
    "evaluation_date": "2026-08-26",
}

if RAGAS_AVAILABLE and 'comparison_results' in dir():
    for name, res in comparison_results.items():
        export_data[f"scores_{name}"] = {
            "faithfulness": float(res.get("faithfulness", 0)),
            "answer_relevancy": float(res.get("answer_relevancy", 0)),
            "context_precision": float(res.get("context_precision", 0)),
            "context_recall": float(res.get("context_recall", 0)),
        }

json_path = output_path / "ragas_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)
print(f"[export] Saved results to {json_path}")

[export] Saved results to D:\projects\python\MLCourse\03_agentic_ai\29_rag_evaluation\ragas_results.json


In [31]:
# ## Summary
#
# - RAGAS provides standardized, repeatable RAG evaluation metrics
# - **Faithfulness** checks answer grounding in context
# - **Answer Relevancy** checks question alignment
# - **Context Precision** checks retrieval quality
# - **Context Recall** checks retrieval coverage
# - Always create reference answers for reliable evaluation
# - Compare strategies side-by-side using the same test dataset